# Setup Instructions to run this notebook
1. Create conda virtual environment with Python 3.8. It is important to have version Python 3.8.
2. Install the below and run this notebook in the created conda environment. 
3. Clone our project repository 
4. Also clone the following: git clone https://github.com/chrismattmann/tika-img-similarity
5. pip install tika editdistance (https://github.com/chrismattmann/tika-similarity?tab=readme-ov-file)
6. Install Java Development Kit
7. Make sure to put the cloned tika-img-similarity folder in the project repository folder. Make sure it is in the base of project repository folder, and not nested inside another folder 
8. Install ETLLib (https://github.com/chrismattmann/etllib)

In [1]:
import os
import sys
import json
import pandas as pd

In [2]:
# Adding the bin directory inside my conda environment to PATH variable so that python 3.8 and etllib commands can be found and accessed
# User should do this on their end as well if necessary
os.environ["PATH"] = "/Users/kirthichillakanti/miniconda3/envs/dsci550_hw_1/bin:" + os.environ["PATH"]

In [3]:
# Checking if previous step worked
!echo $PATH

/Users/kirthichillakanti/miniconda3/envs/dsci550_hw_1/bin:/Users/kirthichillakanti/miniconda3/envs/dsci550_hw_1/bin:/usr/local/mysql/bin:/opt/homebrew/opt/openjdk/bin:/opt/homebrew/opt/openjdk/bin:/opt/homebrew/bin:/Users/kirthichillakanti/miniconda3/bin:/Users/kirthichillakanti/miniconda3/condabin:/opt/homebrew/bin:/opt/homebrew/sbin:/usr/local/bin:/System/Cryptexes/App/usr/bin:/usr/bin:/bin:/usr/sbin:/sbin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/local/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/bin:/var/run/com.apple.security.cryptexd/codex.system/bootstrap/usr/appleinternal/bin:/Library/TeX/texbin


In [4]:
# Checking python verison
!python --version

Python 3.8.20


In [5]:
with open("../conf/colheaders.conf", "r") as file:
    lines = file.readlines()
    print(lines)

['city\n', 'country\n', 'description\n', 'location\n', 'state\n', 'state_abbrev\n', 'longitude\n', 'latitude\n', 'city_longitude\n', 'city_latitude\n', 'date_occured']


In [6]:
# Getting the path to scripts directory
# Creating a path for the output json file when we do tsvtojson
# Storing path of tsv file in tsv_data_file_path variable
scripts_dir =os.getcwd()
json_file_path = os.path.join(scripts_dir, "..", "output_json_file.json")
tsv_data_file_path = os.path.join(scripts_dir, "..", "data", "haunted_places.tsv")

In [8]:
!tsvtojson -t {tsv_data_file_path} -j {json_file_path} -c ../conf/colheaders.conf -o haunted_places -s 0.8 -v -e ../conf/encoding.conf

['utf-8', 'us-asci']
['city', 'country', 'description', 'location', 'state', 'state_abbrev', 'longitude', 'latitude', 'city_longitude', 'city_latitude', 'date_occured']
Deduping list of structs. Count: [10993]
After dedup. Count: [10993]
Near duplicates detection.
Filtered 0 near duplicates.
After near duplicates. Count: [10992]
Writing output file: [/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../output_json_file.json]


In [9]:
# creating a path to a directory called all_json_files to put all the individual json files
parent_of_scripts = os.path.join(scripts_dir, "..")
folder_path = os.path.join(parent_of_scripts, "all_json_files")
os.makedirs(folder_path, exist_ok = True)

In [10]:
def split_aggregated_json(aggregated_json_path, output_base_dir, files_per_folder=100):
    """
    Reads an aggregated JSON file (with an outer wrapper) and writes each row as its own JSON file.
    Organizes the files into subfolders, each containing `files_per_folder` files.

    Args:
        aggregated_json_path (str): Path to the aggregated JSON file.
        output_base_dir (str): Directory where individual JSON files and subfolders will be created.
        files_per_folder (int): Number of JSON files per subfolder.
    """
    # Ensure the output base directory exists.
    if not os.path.exists(output_base_dir):
        os.makedirs(output_base_dir)

    # Load the aggregated JSON file.
    with open(aggregated_json_path, 'r', encoding='utf-8') as infile:
        data = json.load(infile)

    # If your aggregated JSON file is wrapped in an outer object (e.g., {"haunted_places": [ ... ]})
    # then get the list of rows. Otherwise, assume data is the list.
    if isinstance(data, dict):
        # Change "haunted_places" to whatever key your aggregated JSON uses.
        rows = data.get("haunted_places", [])
    else:
        rows = data

    # Loop through each row and write each one to its own file.
    for i, row in enumerate(rows):
        # Determine subfolder index (starting at 1).
        folder_index = (i // files_per_folder) + 1
        subfolder_name = f"dir_{folder_index:03d}"
        subfolder_path = os.path.join(output_base_dir, subfolder_name)

        # Create the subfolder if it doesn't exist.
        if not os.path.exists(subfolder_path):
            os.makedirs(subfolder_path)

        # Create a filename for the JSON file (e.g., row_000001.json).
        json_file_name = f"row_{i:06d}.json"
        json_file_path = os.path.join(subfolder_path, json_file_name)

        # Write the row data as a JSON file.
        with open(json_file_path, 'w', encoding='utf-8') as outfile:
            json.dump(row, outfile, indent=2)

        print(f"Created {json_file_path}")

    print(f"Processed {len(rows)} rows into individual JSON files in '{output_base_dir}'.")


# Example usage:
aggregated_json_path = json_file_path  # Path to your aggregated JSON file.
output_base_dir = folder_path          # Change as needed.
split_aggregated_json(aggregated_json_path, output_base_dir, files_per_folder=100)

Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000000.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000001.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000002.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000003.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000004.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000005.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000006.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_001/row_000007.json
Created /Users/kirthichi

Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002569.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002570.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002571.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002572.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002573.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002574.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002575.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_026/row_002576.json
Created /Users/kirthichi

Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005056.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005057.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005058.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005059.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005060.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005061.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005062.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_051/row_005063.json
Created /Users/kirthichi

Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007337.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007338.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007339.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007340.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007341.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007342.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007343.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_074/row_007344.json
Created /Users/kirthichi

Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009715.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009716.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009717.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009718.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009719.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009720.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009721.json
Created /Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../all_json_files/dir_098/row_009722.json
Created /Users/kirthichi

In [23]:
## RUN THESE IN TERMINAL

#pkill -f tika-server.jar
#java -jar ~/tika-server.jar
os.environ["TIKA_SERVER_JAR"] = "file:////Users/kirthichillakanti/tika-server-standard-3.1.0.jar"

In [24]:
import subprocess

# Storing the path to the cosine_similarity.py file in a variable
cosine_similarity_py_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "distance", "cosine_similarity.py")

# Storing the path to the jaccard_similarity.py file in a variable
jaccard_similarity_py_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "distance", "jaccard_similarity.py")

# Storing the path to the edit_distance.py file in a variable
edit_distance_py_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "distance", "edit-value-similarity.py")


# Added execution permission to cosine_similarity.py file so that it can be executed
subprocess.run(['chmod', '+x', cosine_similarity_py_path]) 
# Added execution permission to jaccard_similarity.py file so that it can be executed
subprocess.run(['chmod', '+x', jaccard_similarity_py_path])
# Added execution permission to edit_distance.py file so that it can be executed
subprocess.run(['chmod', '+x', edit_distance_py_path])

CompletedProcess(args=['chmod', '+x', '/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../tika-img-similarity/tikasimilarity/distance/edit-value-similarity.py'], returncode=0)

In [25]:
# The following functions are for cosine similarity, jaccard similarity, and edit_distance
# These functions take an integer as input.
# This integer input represents the directory number which contains a specific subset of data
# So we will run these similarity functions on a specific subset of data at a time

def cosine_similarity(i:int):
    # d_name stores the name of the directory that has the subset of data user is focused on
    d_name = f"dir_{i:03d}"
    
    # Creates a folder for the certain subset of data that we call this function on. 
    # This folder will contain the cosine_similarity matrix
    results_folder_path = os.path.join(parent_of_scripts, f"dir_{i}")
    os.makedirs(results_folder_path, exist_ok = True)
    
    # Creating a path for the output csv file
    output_csv_path = os.path.join(results_folder_path, "cosine_similarity_matrix.csv")
    
    # Executed the command to run the cosine_similarity.py on the subset of data
    subprocess.run(['python', cosine_similarity_py_path, '--inputDir', f"{folder_path}/{d_name}", '--outCSV', output_csv_path])
    
    # Returns the results_folder_path for use later on, when we want to put the clustering json/html files in the same folder.  
    return results_folder_path 

def jaccard_similarity(i:int):
    d_name = f"dir_{i:03d}"
    
    # Creates a folder for the certain subset of data that we call this function on. 
    # This folder will contain the jaccard similarity matrix
    results_folder_path = os.path.join(parent_of_scripts, f"dir_{i}")
    os.makedirs(results_folder_path, exist_ok = True)
    
    # Creating a path for the output csv file
    output_csv_path = os.path.join(results_folder_path, "jaccard_similarity_matrix.csv")
    
    # Executed the command to run the jaccard_similarity.py on the subset of data
    subprocess.run(['python', jaccard_similarity_py_path, '--inputDir', f"{folder_path}/{d_name}", '--outCSV', output_csv_path])

    # Returns the results_folder_path for use later on, when we want to put the clustering json/html files in the same folder. 
    return results_folder_path

def edit_distance(i:int):
    d_name = f"dir_{i:03d}"
    # Creates a folder for the certain subset of data that we call this function on. 
    # This folder will contain the edit distance similarity matrix
    results_folder_path = os.path.join(parent_of_scripts, f"dir_{i}")
    os.makedirs(results_folder_path, exist_ok = True)
    
    # Creating a path for the output csv file
    output_csv_path = os.path.join(results_folder_path, "edit_distance_matrix.csv")
    
    # Executed the command to run the edit_distance.py on the subset of data
    subprocess.run(['python', edit_distance_py_path, '--inputDir', f"{folder_path}/{d_name}", '--outCSV', output_csv_path])

    # Returns the results_folder_path for use later on, when we want to put the clustering json/html files in the same folder. 
    return results_folder_path

In [28]:
results_folder_path = edit_distance(1)

2025-03-13 09:47:44,874 [MainThread  ] [INFO ]  Retrieving file:////Users/kirthichillakanti/tika-server-standard-3.1.0.jar to /var/folders/0f/_8hrqr0s5s3bkt_029z1q3x80000gn/T/tika-server.jar.
2025-03-13 09:47:44,953 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...
2025-03-13 09:47:49,959 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...
2025-03-13 09:47:54,962 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...


Accepting all MIME Types.....


2025-03-13 09:47:59,965 [MainThread  ] [ERROR]  Tika startup log message not received after 3 tries.
2025-03-13 09:47:59,967 [MainThread  ] [ERROR]  Failed to receive startup confirmation from startServer.
Traceback (most recent call last):
  File "/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../tika-img-similarity/tikasimilarity/distance/edit-value-similarity.py", line 268, in <module>
    computeScores(args.inputDir, args.outCSV, args.accept, args.allKeys)
  File "/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../tika-img-similarity/tikasimilarity/distance/edit-value-similarity.py", line 74, in computeScores
    file2_parsedData = parser.from_file(file2)
  File "/Users/kirthichillakanti/miniconda3/envs/dsci550_hw_1/lib/python3.8/site-packages/tika/parser.py", line 40, in from_file
    output = parse1(service, filename, serverEndpoint, headers=headers, config_path=config_path, requestOptions=requestOptions)
  File "/Users/kirthichill

In [27]:
results_folder_path = cosine_similarity(1)

2025-03-13 09:47:06,456 [MainThread  ] [INFO ]  Retrieving file:////Users/kirthichillakanti/tika-server-standard-3.1.0.jar to /var/folders/0f/_8hrqr0s5s3bkt_029z1q3x80000gn/T/tika-server.jar.
2025-03-13 09:47:06,539 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...
2025-03-13 09:47:11,545 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...
2025-03-13 09:47:16,547 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...


Accepting all MIME Types.....


2025-03-13 09:47:21,552 [MainThread  ] [ERROR]  Tika startup log message not received after 3 tries.
2025-03-13 09:47:21,553 [MainThread  ] [ERROR]  Failed to receive startup confirmation from startServer.
Traceback (most recent call last):
  File "/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../tika-img-similarity/tikasimilarity/distance/cosine_similarity.py", line 123, in <module>
    computeScores(args.inputDir, args.outCSV, args.accept)
  File "/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../tika-img-similarity/tikasimilarity/distance/cosine_similarity.py", line 64, in computeScores
    file1_parsedData = parser.from_file(file1)
  File "/Users/kirthichillakanti/miniconda3/envs/dsci550_hw_1/lib/python3.8/site-packages/tika/parser.py", line 40, in from_file
    output = parse1(service, filename, serverEndpoint, headers=headers, config_path=config_path, requestOptions=requestOptions)
  File "/Users/kirthichillakanti/miniconda3/envs

In [26]:
results_folder_path = jaccard_similarity(1)

2025-03-13 09:46:23,862 [MainThread  ] [INFO ]  Retrieving file:////Users/kirthichillakanti/tika-server-standard-3.1.0.jar to /var/folders/0f/_8hrqr0s5s3bkt_029z1q3x80000gn/T/tika-server.jar.
2025-03-13 09:46:23,927 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...
2025-03-13 09:46:28,933 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...
2025-03-13 09:46:33,937 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...


Accepting all MIME Types.....


2025-03-13 09:46:38,939 [MainThread  ] [ERROR]  Tika startup log message not received after 3 tries.
2025-03-13 09:46:38,940 [MainThread  ] [ERROR]  Failed to receive startup confirmation from startServer.
Traceback (most recent call last):
  File "/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../tika-img-similarity/tikasimilarity/distance/jaccard_similarity.py", line 82, in <module>
    computeScores(args.inputDir, args.outCSV, args.accept)
  File "/Users/kirthichillakanti/Documents/GitHub/DSCI-550-Assignment-1/scripts/../tika-img-similarity/tikasimilarity/distance/jaccard_similarity.py", line 61, in computeScores
    f2MetaData = parser.from_file(file2)["metadata"]
  File "/Users/kirthichillakanti/miniconda3/envs/dsci550_hw_1/lib/python3.8/site-packages/tika/parser.py", line 40, in from_file
    output = parse1(service, filename, serverEndpoint, headers=headers, config_path=config_path, requestOptions=requestOptions)
  File "/Users/kirthichillakanti/minicond

# Example of clustering on dir_001 using cosine similarity matrix

In [40]:
# Run edit-cosine-circle-packing.py 
# Puts the resulting output file into the same folder that contains the similarity matrix for this particular subset of data
# User can replace the inputCSV file to cosine_similarity_matrix.csv or jaccard_similarity_matrix.csv
edit_cosine_circle_packing_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "cluster", "edit-cosine-circle-packing.py")
!chmod +x {edit_cosine_circle_packing_path}
!cd {results_folder_path} && {edit_cosine_circle_packing_path} --inputCSV cosine_similarity_matrix.csv --cluster 2

In [41]:
# Run edit-cosine-cluster.py
# Puts the resulting output file into the same folder that contains the similarity matrix for this particular subset of data
# User can replace the inputCSV file to cosine_similarity_matrix.csv or jaccard_similarity_matrix.csv
edit_cosine_cluster_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "cluster", "edit-cosine-cluster.py")
!chmod +x {edit_cosine_cluster_path}
!cd {results_folder_path} && {edit_cosine_cluster_path} --inputCSV cosine_similarity_matrix.csv --cluster 2

In [42]:
# Run generateLevelCluster.py
# Puts the resulting output file into the same folder that contains the similarity matrix for this particular subset of data
generateLevelCluster_path = os.path.join(scripts_dir, "..", "tika-img-similarity", "tikasimilarity", "cluster", "generateLevelCluster.py")
!chmod +x {generateLevelCluster_path}
!cd {results_folder_path} && {generateLevelCluster_path}

In [32]:
# Copy the files from etllib/html to results_folder_path which contains all the matrices and clustering files
import shutil
etllib_html_path = os.path.expanduser('~/etllib/html')
files = os.listdir(etllib_html_path)

for file in files:
    source = os.path.join(etllib_html_path, file)
    destination = os.path.join(results_folder_path, file)
    shutil.copy(source, destination)

In [43]:
# Start a simple HTTP server to serve the files
!cd {results_folder_path} && python -m http.server 8082

Serving HTTP on :: port 8082 (http://[::]:8082/) ...
::1 - - [13/Mar/2025 10:07:31] "GET /levelCluster-d3.html HTTP/1.1" 304 -
::1 - - [13/Mar/2025 10:07:31] "GET /circle.json HTTP/1.1" 200 -
^C

Keyboard interrupt received, exiting.
